# 03 — Neural Style Transfer
Compare three methods: Gatys (VGG-19 optimization), Johnson (fast residual), AdaIN.

In [ ]:
import sys
sys.path.insert(0, '..')

import torch
import matplotlib.pyplot as plt
from PIL import Image
from torchvision import transforms

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

In [ ]:
# ── Helper functions ──
def load_img(path, size=256):
    img = Image.open(path).convert('RGB').resize((size, size))
    t = transforms.ToTensor()(img).unsqueeze(0)
    return t.to(device)

def show_imgs(imgs, titles, figsize=(16, 4)):
    fig, axes = plt.subplots(1, len(imgs), figsize=figsize)
    for ax, img, title in zip(axes, imgs, titles):
        if isinstance(img, torch.Tensor):
            img = transforms.ToPILImage()(img.squeeze(0).clamp(0,1).cpu())
        ax.imshow(img); ax.set_title(title); ax.axis('off')
    plt.tight_layout(); plt.show()

# ── Load sample images (from DTD or provide your own) ──
from pathlib import Path
import random

dtd = Path('../data/dtd/images')
if dtd.exists():
    cats = sorted(dtd.iterdir())
    content_path = str(random.choice(list(cats[0].glob('*.jpg'))))
    style_path   = str(random.choice(list(cats[10].glob('*.jpg'))))
else:
    print('DTD not found — set content_path and style_path manually')
    content_path = 'content.jpg'
    style_path   = 'style.jpg'

content_img = load_img(content_path, 256)
style_img   = load_img(style_path, 256)
show_imgs([content_img, style_img], ['Content', 'Style'])

In [ ]:
# ── Method 1: Gatys VGG-19 optimization ──
from models.vgg_style_transfer import VGGStyleTransfer
import time

nst = VGGStyleTransfer(content_weight=1.0, style_weight=1e6, tv_weight=1e-4, device=str(device))

t0 = time.time()
gatys_result = nst.transfer(content_img, style_img, num_steps=200)
t_gatys = time.time() - t0

show_imgs([content_img, style_img, gatys_result], 
          ['Content', 'Style', f'Gatys ({t_gatys:.1f}s)'])
print(f'Gatys time: {t_gatys:.1f}s')

In [ ]:
# ── Method 2: AdaIN (fast, no style-specific training needed) ──
from models.adain import AdaINStyleTransfer

adain = AdaINStyleTransfer(device=str(device))

for alpha in [0.25, 0.5, 0.75, 1.0]:
    result = adain.transfer(content_img, style_img, alpha=alpha)
    print(f'alpha={alpha}', end=' ')

# Show alpha sweep
alphas = [0.0, 0.25, 0.5, 0.75, 1.0]
results = [adain.transfer(content_img, style_img, alpha=a) for a in alphas]
show_imgs(results, [f'α={a}' for a in alphas], figsize=(20, 4))

In [ ]:
# ── Gram matrix visualization ──
from models.vgg_style_transfer import VGGFeatureExtractor, gram_matrix

extractor = VGGFeatureExtractor(['relu1_1', 'relu2_1', 'relu3_1', 'relu4_1']).to(device)

mean = torch.tensor([0.485,0.456,0.406], device=device).view(1,3,1,1)
std  = torch.tensor([0.229,0.224,0.225], device=device).view(1,3,1,1)

with torch.no_grad():
    feats = extractor((content_img - mean) / std)

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, (name, feat) in zip(axes, feats.items()):
    g = gram_matrix(feat)[0].cpu().numpy()
    ax.imshow(g[:32, :32], cmap='viridis')
    ax.set_title(f'Gram @ {name}\n{feat.shape[1]} channels')
    ax.axis('off')
plt.suptitle('Gram Matrices (top-left 32×32 block)')
plt.tight_layout(); plt.show()